In [1]:
# ================================================================
#   E-Commerce Data Analysis Project
#   Complete EDA + Advanced EDA  (Data Analyst + Data Scientist)
#   File   : eda_complete.py
#   Input  : featured_superstore.csv
#   Output : charts saved in /charts/ folder
#
#   FLOW OF A REAL DATA SCIENTIST:
#   1. Understand the data shape
#   2. Univariate Analysis    → one column at a time
#   3. Bivariate Analysis     → two columns together
#   4. Multivariate Analysis  → many columns together
#   5. Time Series Analysis   → trends over time
#   6. Customer Analysis      → RFM (who are the best customers?)
#   7. Product Analysis       → what sells, what doesn't
#   8. Correlation Analysis   → which features relate to each other
#   9. Outlier Detection      → find unusual data points
#  10. Business Insights      → final conclusions
# ================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')

# ----------------------------------------------------------------
# GLOBAL SETTINGS
# ----------------------------------------------------------------
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize']  = (12, 5)
plt.rcParams['axes.titlesize']  = 14
plt.rcParams['axes.labelsize']  = 11

os.makedirs("charts", exist_ok=True)   # All charts saved here

print("=" * 60)
print("  E-COMMERCE COMPLETE EDA  ")
print("=" * 60)


# ================================================================
# SECTION 1 — LOAD DATA & BASIC UNDERSTANDING
# ================================================================

df = pd.read_csv("featured_superstore.csv")
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])

print("\n--- SECTION 1: Basic Understanding ---")
print(f"Rows         : {df.shape[0]}")
print(f"Columns      : {df.shape[1]}")
print(f"Date Range   : {df['Order Date'].min().date()}  to  {df['Order Date'].max().date()}")
print(f"Total Sales  : ${df['Sales'].sum():,.2f}")
print(f"Avg Sales    : ${df['Sales'].mean():,.2f}")
print(f"Customers    : {df['Customer Name'].nunique()}")
print(f"Products     : {df['Product Name'].nunique()}")
print(f"Cities       : {df['City'].nunique()}")
print(f"States       : {df['State'].nunique()}")

print("\n--- Data Types ---")
print(df.dtypes)

print("\n--- Missing Values ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Statistical Summary (Numerical Columns) ---")
print(df[['Sales', 'Days to Ship', 'Customer Order Count',
          'Customer Total Sales', 'Product Order Count']].describe().round(2))


# ================================================================
# SECTION 2 — UNIVARIATE ANALYSIS
# One column at a time — understand its distribution
# ================================================================

print("\n\n--- SECTION 2: Univariate Analysis ---")

# ── Chart 1: Sales Distribution (Histogram) ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Sales'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Sales Distribution (All Orders)')
axes[0].set_xlabel('Sales Amount ($)')
axes[0].set_ylabel('Number of Orders')

# Most orders are small — zoom in below $1000 to see clearly
df_zoom = df[df['Sales'] < 1000]
axes[1].hist(df_zoom['Sales'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Sales Distribution (Under $1000 — Zoomed)')
axes[1].set_xlabel('Sales Amount ($)')
axes[1].set_ylabel('Number of Orders')

plt.suptitle('UNIVARIATE: Sales Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/01_sales_distribution.png", dpi=150)
plt.close()
print("✅ Chart 1 saved — Sales Distribution")

# ── Chart 2: Category, Region, Segment counts ────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df['Category'].value_counts().plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Orders by Category')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=15)

df['Region'].value_counts().plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Orders by Region')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

df['Segment'].value_counts().plot(kind='bar', ax=axes[2], color='goldenrod', edgecolor='white')
axes[2].set_title('Orders by Segment')
axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('UNIVARIATE: Orders by Category / Region / Segment', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/02_category_region_segment.png", dpi=150)
plt.close()
print("✅ Chart 2 saved — Category / Region / Segment")

# ── Chart 3: Ship Mode & Shipping Speed ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['Ship Mode'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%',
                                    startangle=90, colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[0].set_title('Ship Mode Distribution')
axes[0].set_ylabel('')

df['Shipping Speed'].value_counts().plot(kind='bar', ax=axes[1],
                                          color=['green','orange','blue','red'], edgecolor='white')
axes[1].set_title('Shipping Speed (Days Group)')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('UNIVARIATE: Shipping Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/03_shipping_analysis.png", dpi=150)
plt.close()
print("✅ Chart 3 saved — Shipping Analysis")

# ── Chart 4: Sales Category & Customer Value ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order = ['Low', 'Medium', 'High', 'Very High']
sales_cat = df['Sales Category'].value_counts().reindex(order)
sales_cat.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Orders by Sales Category')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

cv_order = ['Low Value', 'Medium Value', 'High Value']
cust_val = df['Customer Value'].value_counts().reindex(cv_order)
cust_val.plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Orders by Customer Value')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('UNIVARIATE: Sales Category & Customer Value', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/04_sales_customer_value.png", dpi=150)
plt.close()
print("✅ Chart 4 saved — Sales Category & Customer Value")


# ================================================================
# SECTION 3 — BIVARIATE ANALYSIS
# Two columns together — find relationships
# ================================================================

print("\n\n--- SECTION 3: Bivariate Analysis ---")

# ── Chart 5: Sales by Category & Region ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
cat_sales.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Total Sales by Category')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=0)

reg_sales = df.groupby('Region')['Sales'].sum().sort_values(ascending=False)
reg_sales.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Total Sales by Region')
axes[1].set_xlabel('')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('BIVARIATE: Sales by Category & Region', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/05_sales_category_region.png", dpi=150)
plt.close()
print("✅ Chart 5 saved — Sales by Category & Region")

# ── Chart 6: Sales by Segment & Ship Mode ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_sales = df.groupby('Segment')['Sales'].sum().sort_values(ascending=False)
seg_sales.plot(kind='bar', ax=axes[0], color='goldenrod', edgecolor='white')
axes[0].set_title('Total Sales by Segment')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=0)

ship_sales = df.groupby('Ship Mode')['Sales'].sum().sort_values(ascending=False)
ship_sales.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='white')
axes[1].set_title('Total Sales by Ship Mode')
axes[1].set_xlabel('')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('BIVARIATE: Sales by Segment & Ship Mode', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/06_sales_segment_shipmode.png", dpi=150)
plt.close()
print("✅ Chart 6 saved — Sales by Segment & Ship Mode")

# ── Chart 7: Boxplot — Sales across Categories ───────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='Category', y='Sales', ax=axes[0], palette='Set2')
axes[0].set_title('Sales Spread by Category (Boxplot)')
axes[0].set_ylim(0, 3000)   # Zoom to see spread better

sns.boxplot(data=df, x='Region', y='Sales', ax=axes[1], palette='Set3')
axes[1].set_title('Sales Spread by Region (Boxplot)')
axes[1].set_ylim(0, 3000)

plt.suptitle('BIVARIATE: Sales Spread — Boxplots', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/07_boxplot_sales.png", dpi=150)
plt.close()
print("✅ Chart 7 saved — Boxplots")

# ── Chart 8: Days to Ship by Ship Mode ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_days = df.groupby('Ship Mode')['Days to Ship'].mean().sort_values()
avg_days.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Avg Days to Ship by Ship Mode')
axes[0].set_xlabel('Average Days')

sns.violinplot(data=df, x='Ship Mode', y='Days to Ship', ax=axes[1], palette='muted')
axes[1].set_title('Days to Ship Distribution (Violin)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('BIVARIATE: Shipping Days Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/08_shipping_days.png", dpi=150)
plt.close()
print("✅ Chart 8 saved — Days to Ship")


# ================================================================
# SECTION 4 — MULTIVARIATE ANALYSIS
# Many columns together — deeper patterns
# ================================================================

print("\n\n--- SECTION 4: Multivariate Analysis ---")

# ── Chart 9: Region × Category Heatmap ───────────────────────
pivot = df.pivot_table(values='Sales', index='Region',
                       columns='Category', aggfunc='sum')

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Total Sales ($)'})
ax.set_title('MULTIVARIATE: Sales Heatmap — Region vs Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/09_heatmap_region_category.png", dpi=150)
plt.close()
print("✅ Chart 9 saved — Heatmap Region × Category")

# ── Chart 10: Segment × Category × Sales (Grouped Bar) ───────
pivot2 = df.pivot_table(values='Sales', index='Segment',
                        columns='Category', aggfunc='sum')

fig, ax = plt.subplots(figsize=(12, 5))
pivot2.plot(kind='bar', ax=ax, edgecolor='white')
ax.set_title('MULTIVARIATE: Sales by Segment and Category', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Category')
plt.tight_layout()
plt.savefig("charts/10_segment_category_sales.png", dpi=150)
plt.close()
print("✅ Chart 10 saved — Segment × Category Sales")

# ── Chart 11: Ship Mode × Region × Avg Days (Heatmap) ────────
pivot3 = df.pivot_table(values='Days to Ship', index='Ship Mode',
                        columns='Region', aggfunc='mean').round(1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot3, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Avg Days to Ship'})
ax.set_title('MULTIVARIATE: Avg Shipping Days — Ship Mode vs Region',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/11_heatmap_shipmode_region.png", dpi=150)
plt.close()
print("✅ Chart 11 saved — Heatmap Ship Mode × Region")


# ================================================================
# SECTION 5 — TIME SERIES ANALYSIS
# How does sales change over time?
# ================================================================

print("\n\n--- SECTION 5: Time Series Analysis ---")

# ── Chart 12: Monthly Sales Trend ────────────────────────────
monthly = df.groupby(['Order Year', 'Order Month'])['Sales'].sum().reset_index()
monthly['Date'] = pd.to_datetime(monthly[['Order Year', 'Order Month']].assign(day=1)
                                 .rename(columns={'Order Year':'year','Order Month':'month'}))
monthly = monthly.sort_values('Date')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly['Date'], monthly['Sales'], color='steelblue', linewidth=2, marker='o', markersize=4)
ax.fill_between(monthly['Date'], monthly['Sales'], alpha=0.15, color='steelblue')
ax.set_title('TIME SERIES: Monthly Sales Trend (2015–2018)', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.savefig("charts/12_monthly_sales_trend.png", dpi=150)
plt.close()
print("✅ Chart 12 saved — Monthly Sales Trend")

# ── Chart 13: Yearly Sales Growth ────────────────────────────
yearly = df.groupby('Order Year')['Sales'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
yearly.plot(kind='bar', ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Total Sales by Year')
axes[0].set_xlabel('Year')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=0)

# Year over Year Growth %
yoy = yearly.pct_change() * 100
yoy.dropna().plot(kind='bar', ax=axes[1], color=['red','green','green'], edgecolor='white')
axes[1].set_title('Year over Year Sales Growth (%)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Growth %')
axes[1].tick_params(axis='x', rotation=0)
axes[1].axhline(0, color='black', linewidth=0.8)

plt.suptitle('TIME SERIES: Yearly Sales & Growth', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/13_yearly_sales_growth.png", dpi=150)
plt.close()
print("✅ Chart 13 saved — Yearly Sales & Growth")

# ── Chart 14: Sales by Season & Month ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

season_order = ['Spring', 'Summer', 'Fall', 'Winter']
season_sales = df.groupby('Season')['Sales'].sum().reindex(season_order)
season_sales.plot(kind='bar', ax=axes[0], color=['#55A868','#FF914D','#C44E52','#4C72B0'],
                  edgecolor='white')
axes[0].set_title('Total Sales by Season')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[0].tick_params(axis='x', rotation=0)

month_sales = df.groupby('Order Month')['Sales'].sum()
month_sales.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Total Sales by Month (1=Jan to 12=Dec)')
axes[1].set_xlabel('Month')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('TIME SERIES: Sales by Season & Month', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/14_season_month_sales.png", dpi=150)
plt.close()
print("✅ Chart 14 saved — Season & Month Sales")

# ── Chart 15: Day of Week Sales Pattern ──────────────────────
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_sales  = df.groupby('Order DayOfWeek')['Sales'].sum().reindex(day_order)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(day_sales.index, day_sales.values, color='steelblue', edgecolor='white')
bars[5].set_color('coral')   # Saturday
bars[6].set_color('coral')   # Sunday
ax.set_title('TIME SERIES: Total Sales by Day of Week  (Orange = Weekend)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Total Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.savefig("charts/15_dayofweek_sales.png", dpi=150)
plt.close()
print("✅ Chart 15 saved — Day of Week Sales")


# ================================================================
# SECTION 6 — CUSTOMER ANALYSIS (RFM)
# RFM = Recency, Frequency, Monetary
# Real Data Science technique to find best customers
# ================================================================

print("\n\n--- SECTION 6: Customer RFM Analysis ---")
# Recency  → How recently did the customer order?
# Frequency → How many times did they order?
# Monetary  → How much total money did they spend?

snapshot_date = df['Order Date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('Customer Name').agg(
    Recency   = ('Order Date',    lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Order ID',      'nunique'),
    Monetary  = ('Sales',         'sum')
).reset_index()

# Score each customer 1–4 (4 = best)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1,2,3,4]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# Segment customers
def rfm_segment(score):
    if score >= 10:  return 'Champions'
    elif score >= 8: return 'Loyal Customers'
    elif score >= 6: return 'Potential Loyalists'
    elif score >= 4: return 'At Risk'
    else:            return 'Lost'

rfm['RFM_Segment'] = rfm['RFM_Score'].apply(rfm_segment)

print("\n--- RFM Segment Distribution ---")
print(rfm['RFM_Segment'].value_counts())
print("\n--- Top 10 Champion Customers ---")
print(rfm[rfm['RFM_Segment']=='Champions'][['Customer Name','Recency','Frequency','Monetary']]
      .sort_values('Monetary', ascending=False).head(10).to_string(index=False))

# ── Chart 16: RFM Segment Bar ─────────────────────────────────
seg_order = ['Champions','Loyal Customers','Potential Loyalists','At Risk','Lost']
rfm_counts = rfm['RFM_Segment'].value_counts().reindex(seg_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ecc71','#3498db','#f39c12','#e74c3c','#95a5a6']
rfm_counts.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Customer Segments (RFM)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

# Monetary by segment
rfm_monetary = rfm.groupby('RFM_Segment')['Monetary'].sum().reindex(seg_order)
rfm_monetary.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Total Revenue by RFM Segment')
axes[1].set_xlabel('')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('CUSTOMER ANALYSIS: RFM Segmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/16_rfm_segments.png", dpi=150)
plt.close()
print("✅ Chart 16 saved — RFM Segments")

# ── Chart 17: Top 10 Customers ────────────────────────────────
top10 = df.groupby('Customer Name')['Sales'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 5))
top10.sort_values().plot(kind='barh', ax=ax, color='teal', edgecolor='white')
ax.set_title('CUSTOMER ANALYSIS: Top 10 Customers by Total Sales',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.savefig("charts/17_top10_customers.png", dpi=150)
plt.close()
print("✅ Chart 17 saved — Top 10 Customers")


# ================================================================
# SECTION 7 — PRODUCT ANALYSIS
# ================================================================

print("\n\n--- SECTION 7: Product Analysis ---")

# ── Chart 18: Top 10 Products by Sales ───────────────────────
top_prod = df.groupby('Product Name')['Sales'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
top_prod.sort_values().plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('PRODUCT ANALYSIS: Top 10 Products by Sales',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.savefig("charts/18_top10_products.png", dpi=150)
plt.close()
print("✅ Chart 18 saved — Top 10 Products")

# ── Chart 19: Sub-Category Sales ─────────────────────────────
sub_sales = df.groupby('Sub-Category')['Sales'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 5))
sub_sales.plot(kind='bar', ax=ax, color='coral', edgecolor='white')
ax.set_title('PRODUCT ANALYSIS: Sales by Sub-Category', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig("charts/19_subcategory_sales.png", dpi=150)
plt.close()
print("✅ Chart 19 saved — Sub-Category Sales")

# ── Chart 20: Product Popularity Distribution ─────────────────
fig, ax = plt.subplots(figsize=(10, 5))
pop_order = ['Popular', 'Moderate', 'Rare']
df['Product Popularity'].value_counts().reindex(pop_order).plot(
    kind='pie', ax=ax, autopct='%1.1f%%', startangle=90,
    colors=['#2ecc71','#f39c12','#e74c3c'])
ax.set_title('PRODUCT ANALYSIS: Product Popularity Distribution',
             fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig("charts/20_product_popularity.png", dpi=150)
plt.close()
print("✅ Chart 20 saved — Product Popularity")


# ================================================================
# SECTION 8 — CORRELATION ANALYSIS
# Which numeric features relate to each other?
# ================================================================

print("\n\n--- SECTION 8: Correlation Analysis ---")

num_cols = ['Sales', 'Days to Ship', 'Customer Order Count',
            'Customer Total Sales', 'Product Order Count']
corr_matrix = df[num_cols].corr()

print("\n--- Correlation Matrix ---")
print(corr_matrix.round(3))

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            ax=ax, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'label': 'Correlation'})
ax.set_title('CORRELATION: Numeric Features Correlation Heatmap',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/21_correlation_heatmap.png", dpi=150)
plt.close()
print("✅ Chart 21 saved — Correlation Heatmap")

# ── Chart 22: Scatter — Sales vs Days to Ship ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['Days to Ship'], df['Sales'], alpha=0.3, color='steelblue', s=10)
axes[0].set_title('Sales vs Days to Ship')
axes[0].set_xlabel('Days to Ship')
axes[0].set_ylabel('Sales ($)')

axes[1].scatter(df['Customer Order Count'], df['Sales'], alpha=0.3, color='coral', s=10)
axes[1].set_title('Sales vs Customer Order Count')
axes[1].set_xlabel('Customer Order Count')
axes[1].set_ylabel('Sales ($)')

plt.suptitle('CORRELATION: Scatter Plots', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/22_scatter_plots.png", dpi=150)
plt.close()
print("✅ Chart 22 saved — Scatter Plots")


# ================================================================
# SECTION 9 — OUTLIER DETECTION
# Find unusual data points using IQR and Z-Score methods
# ================================================================

print("\n\n--- SECTION 9: Outlier Detection ---")

# Method 1: IQR (Interquartile Range)
Q1 = df['Sales'].quantile(0.25)
Q3 = df['Sales'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers_iqr = df[(df['Sales'] < lower) | (df['Sales'] > upper)]

print(f"IQR Method  → Lower: ${lower:.2f}  |  Upper: ${upper:.2f}")
print(f"Outliers found (IQR)     : {len(outliers_iqr)} rows")

# Method 2: Z-Score
df['Sales_ZScore'] = np.abs(stats.zscore(df['Sales']))
outliers_z = df[df['Sales_ZScore'] > 3]
print(f"Outliers found (Z-Score) : {len(outliers_z)} rows")
print(f"\nTop 5 Outlier Orders by Sales:")
print(df.nlargest(5, 'Sales')[['Customer Name','Product Name','Category','Sales']].to_string(index=False))

# ── Chart 23: Outlier Visualization ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot(df['Sales'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[0].set_title('Sales Boxplot (Outliers = dots above whisker)')
axes[0].set_ylabel('Sales ($)')

# Highlight outliers in scatter
normal   = df[df['Sales_ZScore'] <= 3]
outlier  = df[df['Sales_ZScore'] > 3]
axes[1].scatter(normal.index,  normal['Sales'],  alpha=0.3, s=8,
                color='steelblue', label='Normal')
axes[1].scatter(outlier.index, outlier['Sales'], alpha=0.9, s=30,
                color='red', label='Outlier')
axes[1].set_title('Outliers Highlighted (Z-Score > 3)')
axes[1].set_xlabel('Row Index')
axes[1].set_ylabel('Sales ($)')
axes[1].legend()

plt.suptitle('OUTLIER DETECTION: Sales Outliers', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("charts/23_outlier_detection.png", dpi=150)
plt.close()
print("✅ Chart 23 saved — Outlier Detection")


# ================================================================
# SECTION 10 — ADVANCED: STATISTICAL TESTS
# Do different groups actually have different sales? (Hypothesis Testing)
# ================================================================

print("\n\n--- SECTION 10: Statistical Tests ---")

# ANOVA Test — Is there a significant difference in Sales across Regions?
# H0 (Null Hypothesis)      : All regions have the same average sales
# H1 (Alternate Hypothesis) : At least one region is different

groups = [df[df['Region']==r]['Sales'].values for r in df['Region'].unique()]
f_stat, p_value = stats.f_oneway(*groups)

print(f"\nANOVA Test — Sales across Regions")
print(f"F-Statistic : {f_stat:.4f}")
print(f"P-Value     : {p_value:.4f}")
if p_value < 0.05:
    print("✅ Result: Significant difference in sales across regions (p < 0.05)")
else:
    print("❌ Result: No significant difference in sales across regions")

# T-Test — Is Consumer segment sales different from Corporate?
consumer  = df[df['Segment']=='Consumer']['Sales']
corporate = df[df['Segment']=='Corporate']['Sales']
t_stat, p_val2 = stats.ttest_ind(consumer, corporate)

print(f"\nT-Test — Consumer vs Corporate Sales")
print(f"T-Statistic : {t_stat:.4f}")
print(f"P-Value     : {p_val2:.4f}")
if p_val2 < 0.05:
    print("✅ Result: Consumer and Corporate sales are significantly different")
else:
    print("❌ Result: No significant difference between Consumer and Corporate sales")


# ================================================================
# SECTION 11 — FINAL BUSINESS INSIGHTS SUMMARY
# ================================================================

print("\n\n" + "=" * 60)
print("  FINAL BUSINESS INSIGHTS SUMMARY")
print("=" * 60)

total_sales   = df['Sales'].sum()
top_category  = df.groupby('Category')['Sales'].sum().idxmax()
top_region    = df.groupby('Region')['Sales'].sum().idxmax()
top_segment   = df.groupby('Segment')['Sales'].sum().idxmax()
top_customer  = df.groupby('Customer Name')['Sales'].sum().idxmax()
top_product   = df.groupby('Product Name')['Sales'].sum().idxmax()
top_subcat    = df.groupby('Sub-Category')['Sales'].sum().idxmax()
busiest_month = df.groupby('Order Month')['Order ID'].count().idxmax()
busiest_day   = df.groupby('Order DayOfWeek')['Order ID'].count().idxmax()
avg_ship_days = df['Days to Ship'].mean()

print(f"""
📦 SALES OVERVIEW
   Total Revenue     : ${total_sales:,.2f}
   Average Order     : ${df['Sales'].mean():,.2f}
   Highest Order     : ${df['Sales'].max():,.2f}

🏆 TOP PERFORMERS
   Best Category     : {top_category}
   Best Sub-Category : {top_subcat}
   Best Region       : {top_region}
   Best Segment      : {top_segment}
   Top Customer      : {top_customer}
   Top Product       : {top_product[:50]}...

📅 TIME INSIGHTS
   Busiest Month     : Month {busiest_month} (November)
   Busiest Day       : {busiest_day}
   Best Year         : 2018

🚚 SHIPPING INSIGHTS
   Avg Ship Days     : {avg_ship_days:.1f} days
   Most Used Mode    : Standard Class
   Fastest Mode      : Same Day

⚠️  OUTLIERS
   IQR Outliers      : {len(outliers_iqr)} high-value orders
   Z-Score Outliers  : {len(outliers_z)} extreme orders

👥 CUSTOMER INSIGHTS
   Total Customers   : {df['Customer Name'].nunique()}
   Champions (RFM)   : {len(rfm[rfm['RFM_Segment']=='Champions'])} customers
   At Risk           : {len(rfm[rfm['RFM_Segment']=='At Risk'])} customers
""")

print("=" * 60)
print("✅ Complete EDA Finished!")
print(f"✅ All charts saved in the 'charts/' folder")
print("=" * 60)

# Clean up temp column
df.drop(columns=['Sales_ZScore'], inplace=True)
df.to_csv("eda_final_superstore.csv", index=False)
print("✅ Final EDA dataset saved as : eda_final_superstore.csv")

  E-COMMERCE COMPLETE EDA  

--- SECTION 1: Basic Understanding ---
Rows         : 4922
Columns      : 33
Date Range   : 2015-01-03  to  2018-12-30
Total Sales  : $1,082,910.82
Avg Sales    : $220.01
Customers    : 793
Products     : 1713
Cities       : 529
States       : 49

--- Data Types ---
Row ID                           int64
Order ID                        object
Order Date              datetime64[ns]
Ship Date               datetime64[ns]
Ship Mode                       object
Customer ID                     object
Customer Name                   object
Segment                         object
Country                         object
City                            object
State                           object
Postal Code                      int64
Region                          object
Product ID                      object
Category                        object
Sub-Category                    object
Product Name                    object
Sales                          float64
Da